<a href="https://colab.research.google.com/github/Payal930-ui/Dairy-Product-Data-Analysis-and-Dashboard/blob/main/Full_superstorre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup The Environment

In [40]:
!pip install pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Superstore Project") \
    .getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Load Dataset

In [41]:
df_basic = spark.read.csv("/content/Sample - Superstore.csv",
                          header=True,
                          inferSchema=True)

df_basic.show(5)
df_basic.printSchema()

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

# Define The Schema

In [42]:
schema = StructType([
    StructField("Row ID", IntegerType(), True),
    StructField("Order ID", StringType(), True),
    StructField("Order Date", StringType(), True),
    StructField("Ship Date", StringType(), True),
    StructField("Ship Mode", StringType(), True),
    StructField("Customer ID", StringType(), True),
    StructField("Customer Name", StringType(), True),
    StructField("Segment", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True),
    StructField("Postal Code", IntegerType(), True),
    StructField("Region", StringType(), True),
    StructField("Product ID", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Sub-Category", StringType(), True),
    StructField("Product Name", StringType(), True),
    StructField("Sales", DoubleType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Discount", DoubleType(), True),
    StructField("Profit", DoubleType(), True)
])

#Load Data with Malformed Handling

In [43]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .schema(schema) \
    .load("/content/Sample - Superstore.csv")

**Separate Good and Bad Data**

In [44]:
if "_corrupt_record" in df.columns:
    bad_df = df.filter(col("_corrupt_record").isNotNull())
    good_df = df.filter(col("_corrupt_record").isNull())
else:
    bad_df = spark.createDataFrame([], df.schema)
    good_df = df

**Save Bad Records**

In [45]:
bad_df.write.mode("overwrite").csv("/content/output/bad_records")

**Clean Data**

In [46]:
df = good_df.drop("_corrupt_record")

# Fix Dates

In [47]:
df = df.withColumn("Order Date", to_date("Order Date", "M/d/yyyy")) \
       .withColumn("Ship Date", to_date("Ship Date", "M/d/yyyy"))

#Handle Null Values

In [48]:
df = df.na.fill({
    "Sales": 0.0,
    "Quantity": 0,
    "Discount": 0.0,
    "Profit": 0.0
})

#Remove Duplicates

In [49]:
df.dropDuplicates()

DataFrame[Row ID: int, Order ID: string, Order Date: date, Ship Date: date, Ship Mode: string, Customer ID: string, Customer Name: string, Segment: string, Country: string, City: string, State: string, Postal Code: int, Region: string, Product ID: string, Category: string, Sub-Category: string, Product Name: string, Sales: double, Quantity: int, Discount: double, Profit: double]

# Extraction From Dates

In [50]:
df = df.withColumn("Year", year("Order Date")) \
       .withColumn("Month", month("Order Date")) \
       .withColumn("Day", dayofmonth("Order Date"))

#Total Sales

In [51]:
df.selectExpr("sum(Sales) as Total_Sales").show()

+------------------+
|       Total_Sales|
+------------------+
|2272449.8562999545|
+------------------+



#Region Wise Sales

In [52]:
df.groupBy("Region") \
  .sum("Sales") \
  .orderBy("sum(Sales)", ascending=False) \
  .show()

+-------+------------------+
| Region|        sum(Sales)|
+-------+------------------+
|   West| 713471.3445000004|
|   East| 672194.0539999981|
|Central| 497800.8728000007|
|  South|388983.58500000037|
+-------+------------------+



#Top 10 Customers

In [53]:
df.groupBy("Customer Name") \
  .sum("Sales") \
  .orderBy("sum(Sales)", ascending=False) \
  .show(10)

+------------------+------------------+
|     Customer Name|        sum(Sales)|
+------------------+------------------+
|       Sean Miller|          25043.05|
|      Tamara Chand|19017.847999999998|
|      Raymond Buch|         15117.339|
|      Tom Ashbrook|          14595.62|
|     Adrian Barton|14355.610999999997|
|      Sanjit Chand|14142.333999999999|
|      Ken Lonsdale|         14071.917|
|      Hunter Lopez|12873.297999999999|
|      Sanjit Engle|12209.438000000002|
|Christopher Conant|         12129.072|
+------------------+------------------+
only showing top 10 rows


#Category Analysis

In [54]:
df.groupBy("Category") \
  .agg({"Sales": "sum", "Profit": "sum"}) \
  .show()

+---------------+-----------------+------------------+
|       Category|       sum(Sales)|       sum(Profit)|
+---------------+-----------------+------------------+
|Office Supplies|703502.9280000031|120632.87839999991|
|      Furniture|733046.8612999996| 19686.42720000003|
|     Technology|835900.0669999964|145388.29659999989|
+---------------+-----------------+------------------+



##**Spark SQL**

In [55]:
df.createOrReplaceTempView("superstore")

**Toatl Profit according to region**

In [56]:
spark.sql("""
SELECT Region, SUM(Profit) as total_profit
FROM superstore
GROUP BY Region
ORDER BY total_profit DESC
""").show()

+-------+------------------+
| Region|      total_profit|
+-------+------------------+
|   West|107303.70150000004|
|   East| 91603.05670000015|
|  South|46650.341000000044|
|Central| 40150.50299999999|
+-------+------------------+



**Total Profit According to customer Name**

In [57]:
spark.sql("""SELECT `Customer Name`, SUM(Profit) as total_profit
FROM superstore
GROUP BY `Customer Name`
ORDER BY total_profit DESC
LIMIT 5
""").show()

+-------------+------------------+
|Customer Name|      total_profit|
+-------------+------------------+
| Tamara Chand| 8964.482600000001|
| Raymond Buch|         6976.0959|
| Sanjit Chand| 5757.411899999999|
| Hunter Lopez|5622.4292000000005|
|Adrian Barton|         5438.9075|
+-------------+------------------+



**Total Sales according to Category**

In [58]:
spark.sql("""SELECT Category, SUM(Sales) as total_sales
FROM superstore
GROUP BY Category
""").show()

+---------------+-----------------+
|       Category|      total_sales|
+---------------+-----------------+
|Office Supplies|703502.9280000031|
|      Furniture|733046.8612999996|
|     Technology|835900.0669999964|
+---------------+-----------------+



#Window Function

**Rank According to Sales**

In [59]:
Sales_rank = Window.partitionBy("Category").orderBy(col("Sales").desc())

df = df.withColumn("Rank", dense_rank().over(Sales_rank))

#SAVE FINAL DATA

In [60]:
df.write.mode("overwrite").parquet("/content/output/superstore_clean")